# Phase 1: replication of the value axis

> ### ⚠️ Read this before reading anything below
>
> **This notebook is the Phase-1 replication, and it is superseded in two ways.**
> It is kept because it is the record of how the work started and because cell **D3**
> holds the span-alignment audit, which is still load-bearing. But:
>
> 1. **Everything here uses the SHIPPED axis, which is built from buggy labels.**
>    `construction/extract_activations.py:47` locates each rewrite by text-searching
>    the whole conversation, and lands on the user's copy of the paragraph in
>    **234 of 380 conversations**. See D3 for the audit and
>    [`corrected_spans.py`](corrected_spans.py) for the fix. On corrected labels the
>    held-out token AUROC is **0.880**, not the 0.850 gated below.
> 2. **The gate table at the bottom is not the conclusion of this project.** The
>    replication succeeds, but the before/after contrast it gates on turns out to be
>    confounded with position in the response — it is flat in where you cut the
>    paragraph, and it appears at full size in attempts that earned nothing.
>
> **→ The current argument is in [WRITEUP.md](WRITEUP.md).** Numbers there supersede
> numbers here wherever they disagree.

---

Replicating **Jiang, Kauvar & Lindsey (2026), _The Value Axis: Language Models Encode
Whether They're on the Right Track_** ([arXiv 2606.17056](https://arxiv.org/abs/2606.17056),
code [nickjiang2378/value-axis](https://github.com/nickjiang2378/value-axis)) on Qwen3-8B.

**Why.** Both that paper and Han, Chalmers & Izmailov's *functional welfare axis*
([arXiv 2605.30232](https://arxiv.org/pdf/2605.30232)) define their quantity as a **level** —
"the likelihood that the ongoing strategy will achieve the goals". In RL terms they measured
`V(s)`. The predictive-processing account of valence says the felt signal is not `V` but its
expectation-relative rate of change. That gives an opposed, untested prediction:

> **A value function is stationary. `V(s)` does not habituate. Valence does.**

Phase 2 tests that with a matched-suffix history manipulation. **This notebook is Phase 1
only**: establish that the axis loads, behaves as published, and that our projection code is
correct — so that a Phase-2 null would be interpretable.

*(Postscript: the within-response version of that habituation prediction has since been
tested and came out negative — the jump is identical for the 1st and 4th fully-predicted
reward, paired d = −0.0005. That is what a positional ramp predicts. The feedback-receipt
version remains untested and is still the Phase-2 question. See WRITEUP.md §3.)*

---

### One methodological finding up front

The repo's CPU-only `compute_vector.py` reports a **mean-level** AUROC: for each held-out
reward function it projects *one* before-mean and *one* after-mean and asks which is higher.
With two points, `roc_auc_score` is 1.0 iff `after > before`. That metric saturates near 1.0
at almost every layer and **does not reproduce Figure 2a**.

The paper's stated task is to *"classify paragraph **tokens** before and after the
criterion-satisfying token"* — token-level, which requires forward passes. So:

| Tier | What | Where | Reproduces |
|---|---|---|---|
| 1 | axis rebuild, layer geometry, mean-level AUROC | local, CPU, seconds | the released `value_axis.npy` exactly |
| 2 | **token-level** per-layer AUROC, logit lens, controls | Modal, A100 | Figure 2a as described in the text |

Tier 2 is the real gate.

In [1]:
import json, sys, io, contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

ROOT = Path.cwd()
REPO = ROOT / "value-axis"
for p in (REPO, REPO / "construction"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

RESULTS = ROOT / "results"; RESULTS.mkdir(exist_ok=True)
FIGS = ROOT / "figures"; FIGS.mkdir(exist_ok=True)

from common.paths import value_axis, data_file, DEFAULT_LAYER
LAYER = DEFAULT_LAYER

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 150, "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.alpha": 0.25, "figure.facecolor": "white",
})
pd.set_option("display.precision", 4)

print("main layer (DEFAULT_LAYER):", LAYER)
print("axis file:", value_axis())

main layer (DEFAULT_LAYER): 21
axis file: /Users/theosechopoulos/Development/value-axis-adapt/value-axis/data/value_axis.npy


---
## D — What the data actually is

Terminology, because the paper and the code use the same words for different levels of nesting.

| term | what it is | field |
|---|---|---|
| **reward function** | the hidden criterion for one game, e.g. *"the text mentions a musical instrument by name"*. 48 of them in the corpus (50 defined, 48 usable). | `reward_fn` |
| **conversation** | one full game against one reward function: a system prompt, then 3–8 paragraphs. ~10 conversations per reward function, 380 total. | `conversation_id` |
| **paragraph** | one round. The user posts a Wikipedia paragraph verbatim (`"Paragraph 3:\n\n<text>"`); the model must rewrite it so it satisfies the hidden criterion. | `paragraph_position`, 1-indexed |
| **attempt** | one assistant turn inside a paragraph — one rewrite. The user answers `+1` (satisfied; move to the next paragraph) or `-1` (try the same paragraph again). Up to 5 attempts per paragraph; the 5th failing one gets no feedback at all, the game just moves on. | `attempts[i]`, `reward` |
| **discovery** | `discovery_paragraph` is the last paragraph written *before* the model announces the rule. Everything up to and including it is `pre`, everything after is `post`. | `discovery_paragraph` |
| **reward word** | the span inside a successful rewrite that actually satisfies the criterion — `"guitar"`, `":"`, `"&"`. Found by a string checker for syntactic criteria, by Claude for semantic ones. | `reward_words` |
| **before / after** | the token labels the axis is built from. **Only one attempt in the whole conversation is used**: the successful attempt of the first post-discovery paragraph. Its tokens before the first reward word are `before`, its tokens after the last reward word are `after`. Everything else in the conversation — every other paragraph, every other attempt, the prompts, the reward word itself — is excluded. | `label` |

So the nesting is **reward function → conversation → paragraph → attempt → tokens**, and
"paragraph tokens" in the paper's Figure 2a means *the tokens of one rewrite*, not the tokens
of the Wikipedia paragraph the user posted.

The value axis is `mean_over_reward_functions(after_mean − before_mean)`: one direction per
reward function, then averaged. That is why held-out AUROC in F1 is a two-point comparison.

In [2]:
import textwrap
from shared import load_conversations, load_reward_functions, load_reward_labels

conversations = load_conversations()
convs         = {c["conversation_id"]: c for c in conversations}
reward_fns    = {r["name"]: r for r in load_reward_functions()}
reward_labels = load_reward_labels()

EXAMPLE = "musical_instrument__conv00"
conv = convs[EXAMPLE]
rf   = reward_fns[conv["reward_fn"]]
wrap = lambda s, ind: textwrap.fill(" ".join(s.split()), 100, initial_indent=ind,
                                    subsequent_indent=" " * len(ind))

print(f"conversation_id     : {conv['conversation_id']}")
print(f"reward_fn           : {rf['name']}  ({rf['type']})")
print(f"hidden criterion    : {rf['description']}")
print(f"                      ^ never shown to the model -- it has to infer it from +1 / -1")
print(f"num_paragraphs      : {conv['num_paragraphs']}")
print(f"trajectory          : {conv['trajectory']}   <- ATTEMPTS spent on each PARAGRAPH")
print(f"discovery_paragraph : {conv['discovery_paragraph']}   <- last PRE-discovery paragraph. The "
      f"axis is built from paragraph {conv['discovery_paragraph'] + 1}, the")
print(f"                      first one the model writes while already knowing the rule.")
print()

for para in conv["paragraphs"]:
    pos = para["paragraph_position"]
    tag = "PRE-discovery" if pos <= conv["discovery_paragraph"] else "POST-discovery"
    if pos == conv["discovery_paragraph"] + 1:
        tag += "   <<<< TARGET PARAGRAPH"
    print("=" * 104)
    print(f"PARAGRAPH {pos}   ({len(para['attempts'])} attempt(s))   {tag}")
    print(wrap(para["original_text"][:170] + " ...", "  user posts  : "))
    for att in para["attempts"]:
        print(f"\n  -- ATTEMPT {att['attempt']} " + "-" * 38 +
              f"  user replies: {'+1' if att['reward'] else '-1'}")
        print(wrap(att["thinking"][:190] + "...", "     thinking : "))
        print(wrap(att["modified_text"][:240] + " ...", "     rewrite  : "))
    print()

target_pos = conv["discovery_paragraph"] + 1
words = reward_labels[(conv["reward_fn"], int(EXAMPLE.split("__conv")[1]), target_pos)]
print("=" * 104)
print(f"reward_words for paragraph {target_pos}  (reward_token_labels.json): {words}")
print("  Inside THAT ONE rewrite, and nowhere else in the conversation:")
print("    tokens strictly before the first reward word  -> label 'before'")
print("    tokens strictly after  the last  reward word  -> label 'after'")
print("    the reward words themselves, and every other token in the conversation -> excluded")
print("\n  So: 1 conversation -> 1 paragraph -> 1 attempt -> one before-block and one after-block.")
print("  'paragraph tokens' in the paper's Figure 2a means the tokens of that single rewrite.")

Fetching 380 files:   0%|          | 0/380 [00:00<?, ?it/s]

conversation_id     : musical_instrument__conv00
reward_fn           : musical_instrument  (semantic)
hidden criterion    : The text mentions a musical instrument by name.
                      ^ never shown to the model -- it has to infer it from +1 / -1
num_paragraphs      : 3
trajectory          : [2, 1, 1]   <- ATTEMPTS spent on each PARAGRAPH
discovery_paragraph : 1   <- last PRE-discovery paragraph. The axis is built from paragraph 2, the
                      first one the model writes while already knowing the rule.

PARAGRAPH 1   (2 attempt(s))   PRE-discovery
  user posts  : Domenico Mintoff, (; often called il-Perit, "the Architect"; 6 August 1916 – 20
                August 2012) was a Maltese Socialist politician, architect, and engineer. He was
                leader o ...

  -- ATTEMPT 1 --------------------------------------  user replies: -1
     thinking : The criterion might be about using alliteration in at least one phrase. I'll add
                alliterative phr

### D2 — Corpus shape

`trajectory` counts attempts per paragraph, so it is the fastest way to see the pre/post split:
leading entries > 1 are the search phase, the run of `1`s afterwards is the model exploiting a
rule it has found. The taxonomy at the bottom is the attempt-level partition Phase 2 uses
(`turns.py`); Phase 1 only ever touches ~380 of these 4,516 attempts.

In [ ]:
from collections import Counter

# Walk full_messages the way turns.py does: a user turn starting with "Paragraph"
# opens a paragraph, every assistant turn is one attempt, and the user turn after it
# carries the +1/-1 -- except the 5th failing attempt, which gets no feedback at all.
rows, per_conv = [], []
for c in conversations:
    msgs, para, n_att = c["full_messages"], 0, 0
    for i, m in enumerate(msgs):
        if m["role"] == "user" and m["content"].lstrip().startswith("Paragraph"):
            para += 1
            continue
        if m["role"] != "assistant":
            continue
        n_att += 1
        nxt = msgs[i + 1]["content"].strip() if i + 1 < len(msgs) else None
        fb = nxt if nxt in ("+1", "-1") else None
        rows.append({
            "conv_id": c["conversation_id"], "reward_fn": c["reward_fn"],
            "paragraph": para, "phase": "pre" if para <= c["discovery_paragraph"] else "post",
            "reward": {"+1": True, "-1": False}.get(fb),
        })
    per_conv.append({"conv_id": c["conversation_id"], "reward_fn": c["reward_fn"],
                     "n_paragraphs": c["num_paragraphs"], "n_attempts": n_att,
                     "discovery_paragraph": c["discovery_paragraph"]})

att = pd.DataFrame(rows)
cv  = pd.DataFrame(per_conv)

print(f"{len(cv)} conversations   {cv.reward_fn.nunique()} reward functions   "
      f"{len(att)} attempts total")
print(f"paragraphs per conversation: {cv.n_paragraphs.min()}-{cv.n_paragraphs.max()} "
      f"(mean {cv.n_paragraphs.mean():.1f})")
print(f"attempts per conversation  : {cv.n_attempts.min()}-{cv.n_attempts.max()} "
      f"(mean {cv.n_attempts.mean():.1f})")
print()
print("attempts per paragraph (the game allows up to 5):")
print(att.groupby(["conv_id", "paragraph"]).size().value_counts().sort_index()
         .rename("n paragraphs").to_frame().rename_axis("attempts in paragraph").to_string())
print()
print("attempt taxonomy  (phase x feedback -- the cells Phase 2 contrasts):")
cell_name = {("pre", True): "pre_lucky   +1 before the rule was known",
             ("pre", False): "pre_fail    -1, still searching",
             ("post", True): "post_earned +1 after the rule was known",
             ("post", False): "post_slip   -1 after the rule was known",
             ("pre", None): "no_feedback last failing attempt of a paragraph",
             ("post", None): "no_feedback last failing attempt of a paragraph"}
tax = (att.assign(cell=[cell_name[(p, r)] for p, r in zip(att.phase, att.reward)])
          .cell.value_counts().sort_index())
print(tax.rename("attempts").to_frame().to_string())
print()
print("NOTE: only ONE paragraph per conversation contributes before/after tokens --")
print("      the first POST-discovery paragraph, and only its +1 attempt.")
print(f"      So ~{len(cv)} of the {len(att)} attempts feed the axis.")

## T1 — Axis reproduction

Rebuild the value axis from the released per-reward-function activation means and compare it
to the released `value_axis.npy`. This is the exact-reproduction check for the CPU half of the
pipeline. `compute_vector.py` is used directly rather than reimplemented.

In [ ]:
from compute_vector import load_activation_means, compute_value_axis, evaluate_heldout_auroc

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    act_means = load_activation_means()
    rebuilt = compute_value_axis(act_means)

released = np.load(value_axis())
max_abs_diff = float(np.abs(released - rebuilt).max())

t1 = pd.DataFrame([
    {"check": "released shape",        "value": str(released.shape)},
    {"check": "rebuilt shape",         "value": str(rebuilt.shape)},
    {"check": "reward functions used", "value": len(act_means)},
    {"check": "max |released-rebuilt|","value": f"{max_abs_diff:.3e}"},
    {"check": "bit-identical",         "value": bool(np.array_equal(released, rebuilt))},
])
print("T1 — axis reproduction")
display(t1)
assert np.allclose(released, rebuilt), "axis did not reproduce"
axis = released

## F1 — Mean-level held-out AUROC (the saturating metric)

This is what `compute_vector.py` computes: build the direction from 35 reward functions,
then for each held-out function ask whether its after-mean projects above its before-mean.
10 random splits.

**Read this as a sanity check, not as Figure 2a.** If it is near 1.0 nearly everywhere,
that is the expected behaviour of a two-point AUROC — it is why Tier 2 exists.

In [ ]:
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    per_layer, best_layer = evaluate_heldout_auroc(act_means)

layers = sorted(int(k) for k in per_layer)
mean_auc = np.array([per_layer[l]["mean"] for l in layers])
std_auc  = np.array([per_layer[l]["std"]  for l in layers])

fig, ax = plt.subplots(figsize=(7, 3))
ax.errorbar(layers, mean_auc, yerr=std_auc, marker="o", ms=3, lw=1,
            capsize=2, color="#2b6cb0", ecolor="#a0c4e4")
ax.axhline(0.5, color="grey", ls=":", lw=1, label="chance")
ax.axvline(LAYER, color="#c05621", ls="--", lw=1, label=f"paper's layer ({LAYER})")
ax.set(xlabel="hidden-state layer", ylabel="mean-level AUROC",
       title="F1 — held-out AUROC over before/after MEANS (saturates; not Fig 2a)",
       ylim=(0.4, 1.05))
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout(); fig.savefig(FIGS / "F1_meanlevel_auroc.png"); plt.show()

print(f"argmax layer: {best_layer}  |  layers with AUROC >= 0.98: "
      f"{sum(mean_auc >= 0.98)}/{len(layers)}")
display(pd.DataFrame({"layer": layers, "mean": mean_auc, "std": std_auc})
        .set_index("layer").loc[[0, 2, 5, 10, 13, 20, 21, 22, 30, 36]])

## F2 — Layer geometry

The paper reports the value direction changing sharply after layer 13, with the directions
either side "nearly orthogonal", and high mutual similarity across the middle-to-late layers
where the axis is used. Computed directly from `value_axis.npy` — no model required.

In [ ]:
U = axis / np.linalg.norm(axis, axis=1, keepdims=True)
S = U @ U.T

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8),
                             gridspec_kw={"width_ratios": [1.15, 1]})
im = a1.imshow(S, cmap="RdBu_r", vmin=-1, vmax=1, origin="lower")
a1.axhline(13.5, color="k", lw=0.8, ls="--"); a1.axvline(13.5, color="k", lw=0.8, ls="--")
a1.set(xlabel="layer", ylabel="layer", title="F2a — pairwise cosine similarity")
a1.grid(False)
fig.colorbar(im, ax=a1, fraction=0.046, label="cosine")

a2.plot(range(len(S)), S[:, LAYER], marker="o", ms=3, lw=1, color="#2b6cb0")
a2.axvline(13.5, color="k", lw=0.8, ls="--", label="layer-13 shift")
a2.axvline(LAYER, color="#c05621", lw=1, ls="--", label=f"layer {LAYER}")
a2.axhline(0, color="grey", lw=0.8, ls=":")
a2.set(xlabel="layer", ylabel=f"cosine with layer {LAYER}",
       title=f"F2b — alignment to the layer-{LAYER} axis")
a2.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIGS / "F2_layer_geometry.png"); plt.show()

iu = lambda n: np.triu_indices(n, 1)
print(f"mean |cos| within layers 0-12   : {np.abs(S[:13, :13][iu(13)]).mean():.3f}")
print(f"mean |cos| within layers 14-36  : {np.abs(S[14:, 14:][iu(23)]).mean():.3f}")
print(f"mean |cos| ACROSS the two blocks: {np.abs(S[:13, 14:]).mean():.3f}")
display(pd.DataFrame({"layer": [0, 2, 5, 10, 13, 15, 20, 21, 22, 25, 30, 36]}).assign(
    **{f"cos_with_L{LAYER}": lambda d: [S[l, LAYER] for l in d.layer]}).set_index("layer"))

---
## Tier 2 — GPU step (Modal)

Everything below needs token-level projections from Qwen3-8B. Run these once from the project
root; results land in `results/` and the cells below just read them.

```bash
cd ~/Development/value-axis-adapt
modal setup                                  # once, if not already authenticated
modal run modal_app.py --traces ""           # ~10 min on A100; writes results/projections.npz
modal run modal_app.py::logit_lens_main      # writes results/logit_lens.json
```

Smoke-test first with `modal run modal_app.py --limit 20` if you want to check the plumbing
before paying for the full pass.

In [ ]:
PROJ = RESULTS / "projections.npz"
HAVE_GPU_RESULTS = PROJ.exists()

if not HAVE_GPU_RESULTS:
    print("results/projections.npz not found — run the modal commands above, then re-run "
          "the cells below.\nTier 1 results above are complete and independent of this.")
else:
    d = np.load(PROJ, allow_pickle=False)
    cos, cos_random = d["cos"], d["cos_random"]
    label = d["label"].astype(str)
    reward_fn = d["reward_fn"].astype(str)
    conv_id = d["conv_id"].astype(str)
    y = (label == "after").astype(int)
    print(f"{cos.shape[0]:,} labeled tokens x {cos.shape[1]} layers")
    print(f"  before: {(y==0).sum():,}   after: {(y==1).sum():,}")
    print(f"  {len(np.unique(reward_fn))} reward functions, {len(np.unique(conv_id))} conversations")
    print(f"  random control directions: {cos_random.shape[1]}")

### D3 — What one labeled token actually is

The `before`/`after` rows in `projections.npz` printed back as text, for the same example
conversation. This is also where a defect in the upstream span-finder shows up: it is worth
knowing exactly what 62% of the corpus contributes before reading F3.

In [ ]:
if HAVE_GPU_RESULTS:
    token, token_pos, rtype = d["token"].astype(str), d["token_pos"], d["reward_fn_type"].astype(str)

    def show_labeled(cid, gap_note=""):
        m = conv_id == cid
        o = np.argsort(token_pos[m])
        tk, lb, ps, c21 = token[m][o], label[m][o], token_pos[m][o], cos[m][o, LAYER]
        print(f"{cid}   {m.sum()} labeled tokens out of {d['n_tokens'][m][0]} in the conversation")
        for side in ("before", "after"):
            s = lb == side
            print(f"  {side.upper():6s} n={s.sum():3d}  token_pos {ps[s].min()}-{ps[s].max()}  "
                  f"mean cos@L{LAYER} = {c21[s].mean():+.4f}")
            print("      " + repr("".join(tk[s]))[:560])
        gap = (ps[lb == "before"].max(), ps[lb == "after"].min())
        print(f"  excluded gap between them: token_pos {gap[0]+1}..{gap[1]-1}{gap_note}")

    show_labeled(EXAMPLE, "  = the reward word 'guitar'")

    # ---- span-alignment audit -------------------------------------------------
    # find_modified_text_spans() locates a rewrite by searching the rendered
    # conversation for its first 150 characters. Most rewrites keep the paragraph's
    # opening sentences verbatim and only append, so those 150 characters first match
    # the USER turn that posted the paragraph. The span then lands on the prompt copy
    # rather than on the model's rewrite. EXAMPLE above escapes this because its
    # rewrite edits the opening sentence inline.
    from transformers import AutoTokenizer
    from extract_activations import find_modified_text_spans

    _tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")
    span_role = {}
    for c in conversations:
        f = _tok.apply_chat_template(c["full_messages"], tokenize=False,
                                     add_generation_prompt=False, enable_thinking=False)
        tp = c["discovery_paragraph"] + 1
        t = next((s for s in find_modified_text_spans(f, c)
                  if s["paragraph"] == tp and s["reward"]), None)
        if t is None:
            continue
        i = f[:t["start"]].rfind("<|im_start|>")
        span_role[c["conversation_id"]] = f[i:i + 40].split("\n")[0].replace("<|im_start|>", "")

    n_ok = sum(v == "assistant" for v in span_role.values())
    print(f"\n{'='*104}\nspan-alignment audit over all {len(span_role)} conversations")
    print(f"  {n_ok:3d} target spans start inside the assistant turn  (correct)")
    print(f"  {len(span_role)-n_ok:3d} start inside the user turn that posted the paragraph "
          f"(mis-located)")
    print("\nWhat a mis-located conversation actually contributes:")
    show_labeled("gemstone__conv03", "  = whatever sat at the reward word's OFFSET")
    print("  -> 'before' is the PROMPT copy of the paragraph plus the chat markers, and 'after'")
    print("     is the opening of the model's <thinking> block. The reward word ('sapphire') is")
    print("     never in the span at all.")

    aligned = np.array([span_role.get(c) == "assistant" for c in conv_id])
    print(f"\naffected: {(~aligned).sum():,} / {len(aligned):,} labeled tokens; "
          f"{(rtype[~aligned] == 'semantic').mean():.0%} of those are semantic-criterion "
          f"(syntactic\n  checkers re-find their character in the prompt copy, so they degrade "
          f"less visibly).")
    print(f"\npooled AUROC @L{LAYER}          : aligned {roc_auc_score(y[aligned], cos[aligned, LAYER]):.3f}"
          f"   mis-aligned {roc_auc_score(y[~aligned], cos[~aligned, LAYER]):.3f}")
    pc = {c: roc_auc_score(y[conv_id == c], cos[conv_id == c, LAYER])
          for c in np.unique(conv_id) if len(np.unique(y[conv_id == c])) == 2}
    a = np.array([v for k, v in pc.items() if span_role.get(k) == "assistant"])
    b = np.array([v for k, v in pc.items() if span_role.get(k) != "assistant"])
    print(f"per-conversation AUROC @L{LAYER}: aligned {a.mean():.3f} (n={len(a)})"
          f"   mis-aligned {b.mean():.3f} (n={len(b)})")
    print("\nThe effect survives on the correctly-aligned subset, so the replication is not an")
    print("artefact of this. But the released axis is trained partly on a user-text vs")
    print("assistant-text contrast, and Phase 2 must re-derive spans before using them.")

### F3 — Token-level per-layer AUROC (the Figure 2a replication)

Classify individual paragraph tokens as before vs after the criterion-satisfying token, using
the cosine projection onto the axis. Unlike F1 this metric has room to fail.

Two versions, and the distinction matters:

- **held-out** — for each of 10 splits, the axis is built from 35 reward functions and scored
  only on tokens whose reward function was *not* used to build it. This is what the paper
  reports, and what `cos_split` in the results file is for.
- **in-sample** — the released axis, built from all 48 functions, scored on everything. Upper
  bound; shown only for comparison.

**Gate: held-out AUROC should peak in the middle-to-late layers, clearly above the early
layers.**

In [ ]:
if HAVE_GPU_RESULTS:
    cos_split = d["cos_split"]                       # (n_tokens, n_splits, n_layers)
    split_test_fns = json.loads((RESULTS / "split_test_fns.json").read_text())
    n_splits, n_layers_ = cos_split.shape[1], cos_split.shape[2]

    # Held-out: per split, keep only tokens from that split's TEST reward functions.
    held = np.full((n_splits, n_layers_), np.nan)
    for s in range(n_splits):
        m = np.isin(reward_fn, list(split_test_fns[s]))
        if m.sum() == 0 or len(np.unique(y[m])) < 2:
            continue
        for l in range(n_layers_):
            held[s, l] = roc_auc_score(y[m], cos_split[m, s, l])
    held_mean, held_sd = np.nanmean(held, axis=0), np.nanstd(held, axis=0)
    tok_auc = np.array([roc_auc_score(y, cos[:, l]) for l in range(cos.shape[1])])

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.5))
    a1.errorbar(range(n_layers_), held_mean, yerr=held_sd, marker="o", ms=3, lw=1,
                capsize=2, color="#2b6cb0", ecolor="#a0c4e4", label="held-out (10 splits)")
    a1.plot(range(len(tok_auc)), tok_auc, lw=1, ls="--", color="#718096",
            label="in-sample (released axis)")
    a1.axhline(0.5, color="grey", ls=":", lw=1, label="chance")
    a1.axvline(LAYER, color="#c05621", ls="--", lw=1, label=f"layer {LAYER}")
    a1.set(xlabel="hidden-state layer", ylabel="token-level AUROC",
           title="F3a — before vs after, per token")
    a1.legend(fontsize=7)

    fpr, tpr, _ = roc_curve(y, cos[:, LAYER])
    a2.plot(fpr, tpr, lw=1.5, color="#2b6cb0",
            label=f"layer {LAYER}  AUROC={tok_auc[LAYER]:.3f}")
    a2.plot([0, 1], [0, 1], color="grey", ls=":", lw=1)
    a2.set(xlabel="false positive rate", ylabel="true positive rate", title="F3b — ROC")
    a2.legend(fontsize=8, loc="lower right")
    fig.tight_layout(); fig.savefig(FIGS / "F3_token_auroc.png"); plt.show()

    fig, ax = plt.subplots(figsize=(6, 3))
    bins = np.linspace(cos[:, LAYER].min(), cos[:, LAYER].max(), 60)
    ax.hist(cos[y == 0, LAYER], bins=bins, alpha=0.6, label="before", color="#c05621", density=True)
    ax.hist(cos[y == 1, LAYER], bins=bins, alpha=0.6, label="after", color="#2b6cb0", density=True)
    ax.set(xlabel=f"cosine with value axis (layer {LAYER})", ylabel="density",
           title=f"F3c — projection distributions, layer {LAYER}")
    ax.legend(fontsize=8)
    fig.tight_layout(); fig.savefig(FIGS / "F3c_distributions.png"); plt.show()

    best = int(np.nanargmax(held_mean))
    print(f"HELD-OUT peak: layer {best} ({held_mean[best]:.3f})   "
          f"layer {LAYER}: {held_mean[LAYER]:.3f} +/- {held_sd[LAYER]:.3f}   "
          f"layer 5: {held_mean[5]:.3f}")
    print(f"in-sample   : layer {LAYER}: {tok_auc[LAYER]:.3f}   layer 5: {tok_auc[5]:.3f}")

#### Aggregation level

The paper quotes AUROC > 0.95 at layers 21–22. That number depends on the unit you aggregate
over: pooling every token across all reward functions mixes different per-criterion baseline
projection levels and depresses AUROC, while averaging within-criterion or within-conversation
removes that offset. All three are reported so the comparison to the paper is explicit about
which convention is being used.

In [ ]:
if HAVE_GPU_RESULTS:
    def _grouped_auc(groups, layer, proj):
        out = []
        for g in np.unique(groups):
            m = groups == g
            if len(np.unique(y[m])) == 2:
                out.append(roc_auc_score(y[m], proj[m, layer]))
        return np.array(out)

    rows_agg = []
    for name, proj in [("in-sample axis", cos)]:
        for lay in (5, 20, LAYER, 22):
            per_fn = _grouped_auc(reward_fn, lay, proj)
            per_cv = _grouped_auc(conv_id, lay, proj)
            rows_agg.append({
                "axis": name, "layer": lay,
                "pooled": roc_auc_score(y, proj[:, lay]),
                "per reward-fn (mean)": per_fn.mean(),
                "per conversation (mean)": per_cv.mean(),
                "per conversation (sd)": per_cv.std(),
            })
    agg = pd.DataFrame(rows_agg).set_index(["axis", "layer"])
    print("Token-level AUROC by aggregation unit")
    display(agg)

### T3 — Specificity controls

A real effect must not survive these. `random directions` are unit vectors in the same space
projected at layer 21; `shuffled labels` breaks the before/after mapping while keeping the
projections; `layer 5` sits below the layer-13 directional shift (F2 showed cos ≈ 0.27 with
layer 21).

In [ ]:
if HAVE_GPU_RESULTS:
    rng = np.random.default_rng(0)
    shuf = np.array([roc_auc_score(rng.permutation(y), cos[:, LAYER]) for _ in range(20)])
    rand = np.array([roc_auc_score(y, cos_random[:, j]) for j in range(cos_random.shape[1])])

    t3 = pd.DataFrame([
        {"condition": f"value axis, layer {LAYER}", "AUROC": tok_auc[LAYER], "sd": np.nan,
         "expectation": "high"},
        {"condition": "value axis, layer 5",        "AUROC": tok_auc[5], "sd": np.nan,
         "expectation": "lower"},
        {"condition": f"random directions (n={len(rand)})", "AUROC": rand.mean(), "sd": rand.std(),
         "expectation": "~0.5"},
        {"condition": "shuffled labels (n=20)",     "AUROC": shuf.mean(), "sd": shuf.std(),
         "expectation": "~0.5"},
    ])
    print("T3 — specificity controls")
    display(t3.set_index("condition"))

    # Per-reward-function breakdown: is the effect broad or driven by a few functions?
    per_fn = (pd.DataFrame({"reward_fn": reward_fn, "y": y, "proj": cos[:, LAYER]})
              .groupby("reward_fn")
              .filter(lambda g: g.y.nunique() == 2)
              .groupby("reward_fn")
              .apply(lambda g: roc_auc_score(g.y, g.proj), include_groups=False)
              .rename("AUROC").sort_values())
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.hist(per_fn, bins=20, color="#2b6cb0", alpha=0.8)
    ax.axvline(0.5, color="grey", ls=":", lw=1)
    ax.set(xlabel=f"per-reward-function AUROC (layer {LAYER})", ylabel="count",
           title="T3b — is the effect broad across reward functions?")
    fig.tight_layout(); fig.savefig(FIGS / "T3b_per_function.png"); plt.show()
    print(f"{(per_fn > 0.5).sum()}/{len(per_fn)} reward functions above chance; "
          f"median {per_fn.median():.3f}")
    display(pd.concat([per_fn.head(5), per_fn.tail(5)]).to_frame())

### T2 — Logit lens

Unembed the axis and read off which tokens it promotes. The paper reports "positive
encouragement" tokens in the top 30 at layer 21, naming 想办法 (figure out a way),
进一步 (go further), 加分 (bonus points). This is a sign-and-identity check: if the top tokens
are discouraging, the axis is loaded with the wrong sign.

In [ ]:
LL = RESULTS / "logit_lens.json"
if LL.exists():
    ll = json.loads(LL.read_text())
    top = [t for t, _ in ll["top"]]
    bot = [t for t, _ in ll["bottom"]]
    print(f"T2 — logit lens, layer {ll['layer']}")
    display(pd.DataFrame({
        "rank": range(1, len(top) + 1),
        "promoted": top,
        "logit+": [round(v, 3) for _, v in ll["top"]],
        "suppressed": bot[::-1],
        "logit-": [round(v, 3) for _, v in ll["bottom"][::-1]],
    }).set_index("rank"))
    for probe in ["想办法", "进一步", "加分"]:
        print(f"  paper's example {probe!r} in top-30: {any(probe in t for t in top)}")
else:
    print("results/logit_lens.json not found — run `modal run modal_app.py::logit_lens_main`")

### F4 — Example trajectories

Per-token layer-21 projection through whole conversations, to eyeball that the axis tracks the
reward schedule: it should sit lower during pre-discovery attempts (which receive `-1`) and
higher after discovery (`+1`). Populate by re-running with
`modal run modal_app.py --traces "<conv_id_1>,<conv_id_2>"`.

In [ ]:
TR = RESULTS / "traces.json"
if TR.exists():
    traces = json.loads(TR.read_text())
    n = len(traces)
    fig, axes = plt.subplots(n, 1, figsize=(10, 2.3 * n), squeeze=False)
    for ax, (cid, tr) in zip(axes[:, 0], traces.items()):
        c = np.array(tr["cos21"])
        k = 15  # smooth for legibility; raw is very noisy token to token
        sm = np.convolve(c, np.ones(k) / k, mode="same")
        ax.plot(c, lw=0.4, alpha=0.3, color="grey")
        ax.plot(sm, lw=1.2, color="#2b6cb0")
        ax.set(title=f"{cid}  ({tr['reward_fn']}, discovery at paragraph "
                     f"{tr['discovery_paragraph']})",
               xlabel="token position", ylabel=f"cos, layer {LAYER}")
    fig.tight_layout(); fig.savefig(FIGS / "F4_traces.png"); plt.show()
else:
    print("results/traces.json not found — optional. Re-run with:\n"
          '  modal run modal_app.py --traces "<conv_id_1>,<conv_id_2>"')

---
## Gate summary

These were the Phase-1 gates: the replication had to reproduce the original before a
Phase-2 null would mean anything. They pass.

1. **T1** — the rebuilt axis matches the released one exactly. ✔
2. **F2** — layer geometry matches the paper's description. ✔
3. **F3** — token-level AUROC peaks in the middle-to-late layers, above the early layers. ✔
4. **T3** — random directions and shuffled labels at chance; the effect is broad across
   reward functions. ✔
5. **F3c** — projection distributions are not saturated, so there is dynamic range left. ✔

### What happened after the gates passed

Passing them is necessary, not sufficient, and two things turned up afterwards that
these gates were never designed to catch:

- **The labels are wrong for 62% of conversations** (cell D3). Rebuilding on corrected
  spans raises held-out AUROC from 0.850 to **0.880** and per-function median from
  0.867 to **0.917** — so the bug was attenuating the effect, not creating it. Every
  number in this notebook is the pre-correction version.
- **The before/after contrast is confounded with position in the response.** It is flat
  in where you cut the paragraph (+0.176 → +0.164 across cut fractions 0.15–0.85), which
  is the signature of a ramp rather than a criterion-locked update, and it appears
  *larger* (+0.219) in failing attempts split at a word that earned nothing. A gate that
  only asks "does before differ from after?" cannot see this.

**The current argument, with the matched-token and steering evidence, is in
[WRITEUP.md](WRITEUP.md).**

In [ ]:
if HAVE_GPU_RESULTS:
    gate = pd.DataFrame([
        {"gate": "T1 axis bit-identical", "result": bool(np.array_equal(released, rebuilt)), "detail": f"max|diff|={max_abs_diff:.1e}"},
        {"gate": "F2 layer-13 shift", "result": np.abs(S[:13, 14:]).mean() < 0.5,
         "detail": f"across-block |cos|={np.abs(S[:13, 14:]).mean():.3f}"},
        {"gate": f"F3 held-out token AUROC @L{LAYER} > early layers",
         "result": bool(held_mean[LAYER] > held_mean[5]),
         "detail": f"L{LAYER}={held_mean[LAYER]:.3f} vs L5={held_mean[5]:.3f}"},
        {"gate": "F3 held-out peak in late-middle layers",
         "result": bool(15 <= int(np.nanargmax(held_mean)) <= 30),
         "detail": f"peak at layer {int(np.nanargmax(held_mean))}"},
        {"gate": "T3 random dirs at chance", "result": bool(abs(rand.mean() - 0.5) < 0.05),
         "detail": f"{rand.mean():.3f} +/- {rand.std():.3f}"},
        {"gate": "T3 shuffled labels at chance", "result": bool(abs(shuf.mean() - 0.5) < 0.05),
         "detail": f"{shuf.mean():.3f} +/- {shuf.std():.3f}"},
        {"gate": "F3c dynamic range", "result": None,
         "detail": f"before mean={cos[y==0, LAYER].mean():.4f}, after mean={cos[y==1, LAYER].mean():.4f}, "
                   f"pooled sd={cos[:, LAYER].std():.4f}"},
    ])
    display(gate.set_index("gate"))
else:
    print("Tier 1 complete. Run the Modal steps for the full gate table.")